In [13]:
import pandas as pd
from pathlib import Path

import numpy as np
from matplotlib.figure import Figure

In [3]:
rgid_mask = "RGID.isin([1,2,3])"

old_hh = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-08-19_90-90-90.xlsx', sheet_name='HHwork').query(rgid_mask)
old_emp = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-08-19_90-90-90.xlsx', sheet_name='EMPwork').query(rgid_mask)
new_hh = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-10-13_90-90-90.xlsx', sheet_name='HHwork').query(rgid_mask)
new_emp = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-10-13_90-90-90.xlsx', sheet_name='EMPwork').query(rgid_mask)

In [ ]:


RG_LABELS = {1: 'Metro', 2: 'Core Cities', 3: 'Larger Cities'}
EXCLUDED_SUBREG_ID = 1401  # NBK Bremerton
# (capacity color, after-scaling color) per run
RUN_COLORS = {'new': ('tab:blue', 'tab:orange'),'old': ('lightsteelblue','navajowhite') }


def _tod_shares(df, cap_col, run):
    """TOD rows of an HHwork/EMPwork table with start/final/increase columns suffixed by ``run``."""
    out = df[(df['is_tod'] == 1) & df['RGID'].isin(RG_LABELS) & (df['subreg_id'] != EXCLUDED_SUBREG_ID)][
        ['nosplit_geo_id', 'RGID', 'name', cap_col, 'target_share']
    ].rename(columns={cap_col: f'start_{run}', 'target_share': f'final_{run}', 'RGID': f'RGID_{run}', 'name': f'name_{run}'})
    out[f'increase_{run}'] = out[f'final_{run}'] - out[f'start_{run}']
    return out


def plot_final_shares(old_df, new_df, cap_col, title, old_name='old', new_name='new', output_dir=None, file_name=None):
    """Plot starting (capacity) and final TOD shares per TOD area for the old and new runs.

    Same layout as ``plot_final_shares`` in ``split_ct_to_hct.py``: stacked bars
    with the capacity share below and the change to the final share on top,
    one panel per regional geography stacked vertically. Old and new runs are drawn side by side
    for each TOD area in different colors.

    Args:
        old_df, new_df: HHwork/EMPwork tables with ``is_tod``, ``RGID``,
            ``nosplit_geo_id``, ``name``, ``cap_col`` and ``target_share`` columns.
        cap_col: Column holding the starting capacity share (``DUcapshare`` or ``EMPcapshare``).
        title: Figure title.
        old_name, new_name: Run names used in the legend.
        output_dir: If given, the image is saved here (created if missing).
        file_name: Image file name; required when ``output_dir`` is given.

    Returns:
        The matplotlib Figure, or None if there was nothing to plot.
    """
    bars = _tod_shares(new_df, cap_col, 'new').merge(
        _tod_shares(old_df, cap_col, 'old'), on='nosplit_geo_id', how='outer'
    )
    if bars.empty:
        return None
    bars['RGID'] = bars['RGID_new'].fillna(bars['RGID_old'])
    bars['name'] = bars['name_new'].fillna(bars['name_old'])
    bars['sort'] = bars['start_new'].fillna(bars['start_old'])

    rg_ids = [rgid for rgid in RG_LABELS if (bars['RGID'] == rgid).any()]
    widths = [(bars['RGID'] == rgid).sum() for rgid in rg_ids]
    fig = Figure(figsize=(max(8, 0.4 * max(widths) + 2), 6 * len(rg_ids)), layout='constrained')
    axes = fig.subplots(len(rg_ids), 1, squeeze=False, sharey=True)
    bar_w = 0.4
    for ax, rgid in zip(axes[:, 0], rg_ids):
        sub = bars[bars['RGID'] == rgid].sort_values('sort', ascending=False)
        x = np.arange(len(sub))
        for run, run_name, offset in [('new', new_name, -bar_w / 2), ('old', old_name, bar_w / 2)]:
            cap_color, scaled_color = RUN_COLORS[run]
            ax.bar(x + offset, sub[f'start_{run}'], width=bar_w, color=cap_color, label=f'{run_name} capacity')
            ax.bar(x + offset, sub[f'increase_{run}'], width=bar_w, bottom=sub[f'start_{run}'],
                   color=scaled_color, label=f'{run_name} after scaling')
        ax.set_xticks(x)
        ax.set_xlim(-0.5, max(widths) - 0.5)  # same bar width in every panel
        ax.set_xticklabels(sub['name'], rotation=90, fontsize=7)
        ax.set_title(RG_LABELS[rgid], fontsize=9)
        ax.set_ylim(0, 100)
        ax.set_ylabel('HCT shares (%)')
    axes[0][0].legend(title='HCT shares', loc='upper right')
    fig.suptitle(title)

    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_dir / file_name, dpi=150)
    return fig


In [ ]:
# Set to a directory (e.g. Path('output/plots')) to also save the images
PLOT_DIR = Path('output/plots')
OLD_NAME = 'old_flu_old_HCTs'
NEW_NAME = 'new_flu_new_HCTs'

today = pd.Timestamp.today().strftime('%Y-%m-%d')
plot_final_shares(old_hh, new_hh, 'DUcapshare', 'Households', old_name=OLD_NAME, new_name=NEW_NAME, output_dir=PLOT_DIR, file_name=f'hct_HH_comparison_{today}.png'),
plot_final_shares(old_emp, new_emp, 'EMPcapshare', 'Employment', old_name=OLD_NAME, new_name=NEW_NAME, output_dir=PLOT_DIR, file_name=f'hct_EMP_comparison_{today}.png'),

,subreg_id,nosplit_geo_id,RGID,name,is_tod,DUtotcapacity,DUnetcapacity,DUcapshare,target_growth_ini,target_growth_final,HH2023,PPH2023,HH2050,PPH2050,target_share
0,1001,1,1,Bellevue,1,81468,51018,75.3,29778,35596,30450,2.162167,66046,2.329417,90.0
1,1,1,1,Bellevue,0,48490,16743,24.7,9773,3955,31747,2.735975,35702,2.761422,10.0
2,1002,2,1,Seattle,1,573197,291476,88.5,92960,94560,281721,1.911608,376281,1.951234,90.0
3,2,2,1,Seattle,0,127543,37960,11.5,12107,10507,89583,2.361564,100090,2.387531,10.0
4,1003,3,2,Auburn,1,17769,9433,30.8,4098,4435,8336,2.760677,12771,2.603658,33.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,1142,142,3,Mukilteo MUGA,1,4830,3967,52.9,2490,2519,863,2.968714,3382,2.267659,53.6
172,142,142,3,Mukilteo MUGA,0,8370,3525,47.1,2212,2183,4845,2.842312,7028,2.589308,46.4
219,1401,401,1,NBK - Bremerton,1,0,0,100.0,3,3,0,0.000000,3,1.200000,100.0
220,401,401,1,NBK - Bremerton,0,0,0,0.0,0,0,0,0.000000,0,0.000000,0.0


In [ ]:
new_hh = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-09-14_90-90-90.xlsx', sheet_name='HHwork').query(rgid_mask)
new_emp = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-09-14_90-90-90.xlsx', sheet_name='EMPwork').query(rgid_mask)

# Set to a directory (e.g. Path('output/plots')) to also save the images
PLOT_DIR = Path('output/plots')
OLD_NAME = 'old_flu_old_HCTs'
NEW_NAME = 'old_flu_new_HCTs'


plot_final_shares(old_hh, new_hh, 'DUcapshare', 'Households', old_name=OLD_NAME, new_name=NEW_NAME, output_dir=PLOT_DIR, file_name='hct_HH_comparison_old_flu.png'),
plot_final_shares(old_emp, new_emp, 'EMPcapshare', 'Employment', old_name=OLD_NAME, new_name=NEW_NAME, output_dir=PLOT_DIR, file_name='hct_EMP_comparison_old_flu.png'),

In [5]:

old_hh = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-10-07_90-90-90.xlsx', sheet_name='HHwork').query(rgid_mask)
old_emp = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-10-07_90-90-90.xlsx', sheet_name='EMPwork').query(rgid_mask)
new_hh = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-10-06_90-90-90.xlsx', sheet_name='HHwork').query(rgid_mask)
new_emp = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-10-06_90-90-90.xlsx', sheet_name='EMPwork').query(rgid_mask)

# Set to a directory (e.g. Path('output/plots')) to also save the images
PLOT_DIR = Path('output/plots')
OLD_NAME = 'new_flu_old_HCTs'
NEW_NAME = 'new_flu_new_HCTs'


plot_final_shares(old_hh, new_hh, 'DUcapshare', 'Households', old_name=OLD_NAME, new_name=NEW_NAME, output_dir=PLOT_DIR, file_name='hct_HH_comparison_new_flu.png'),
plot_final_shares(old_emp, new_emp, 'EMPcapshare', 'Employment', old_name=OLD_NAME, new_name=NEW_NAME, output_dir=PLOT_DIR, file_name='hct_EMP_comparison_new_flu.png'),

(<Figure size 1560x1800 with 3 Axes>,)

In [7]:
old_hh = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-10-14_90-90-90.xlsx', sheet_name='HHwork').query(rgid_mask)
old_emp = pd.read_excel('output/LUVit_ct_by_tod_generator-2026-10-14_90-90-90.xlsx', sheet_name='EMPwork').query(rgid_mask)

new_hh = pd.read_excel(f'output/LUVit_ct_by_tod_generator-2026-10-28_90-90-90.xlsx', sheet_name='HHwork').query(rgid_mask)
new_emp = pd.read_excel(f'output/LUVit_ct_by_tod_generator-2026-10-28_90-90-90.xlsx', sheet_name='EMPwork').query(rgid_mask)

# Set to a directory (e.g. Path('output/plots')) to also save the images
OLD_NAME = '10/14'
NEW_NAME = '10/28'

today = pd.Timestamp.today().strftime('%Y-%m-%d')
plot_final_shares(old_hh, new_hh, 'DUcapshare', 'Households', old_name=OLD_NAME, new_name=NEW_NAME, output_dir='output/plots', file_name=f'hct_HH_comparison_{today}.png'),
plot_final_shares(old_emp, new_emp, 'EMPcapshare', 'Employment', old_name=OLD_NAME, new_name=NEW_NAME, output_dir='output/plots', file_name=f'hct_EMP_comparison_{today}.png'),

(<Figure size 1320x1800 with 3 Axes>,)

In [2]:
with pd.HDFStore('data/pipeline.h5') as store:
    tables = store.keys()
print(tables)

['/adjusted_emp_change_targets', '/adjusted_emp_change_targets_no_res_con', '/adjusted_emp_change_targets_res_con', '/adjusted_king_targets', '/adjusted_total_pop_change_targets', '/adjusted_units_change_targets', '/annual_employment_control_totals', '/annual_employment_control_totals_region', '/annual_household_control_totals', '/annual_household_control_totals_region', '/block_control_area_xwalk', '/blocks', '/borrow_distribution', '/control_areas', '/control_hct', '/control_target_xwalk', '/control_totals', '/current_parcel_control_area_xwalk', '/dec_block_data', '/decennial_by_control_area', '/employment_2018_by_control_area', '/employment_2019_by_control_area', '/employment_2020_by_control_area', '/employment_2023_by_control_area', '/extrapolated_targets', '/king_targets', '/kitsap_targets', '/ofm_parcel_control_area_xwalk', '/ofm_parcelized_2018', '/ofm_parcelized_2018_by_control_area', '/ofm_parcelized_2019', '/ofm_parcelized_2019_by_control_area', '/ofm_parcelized_2020', '/ofm_

In [6]:
luvit_cont_dir = "J:/Projects/LandUseVision/LUV.3/Control_totals/SubJuris_ConTots"
luv_hh = pd.read_excel(f'{luvit_cont_dir}/LUVit_ct_by_tod_generator_90-90-90_2024-10-18_6575_LUVit_Match.xlsx', sheet_name='HHwork').rename(columns={'RGid': 'RGID'}).query(rgid_mask)
luv_emp = pd.read_excel(f'{luvit_cont_dir}/LUVit_ct_by_tod_generator_90-90-90_2024-10-18_6575_LUVit_Match.xlsx', sheet_name='EMPwork').rename(columns={'RGid': 'RGID'}).query(rgid_mask)

In [12]:
def merge_runs(luv_df, new_df, cap_prefix, year_cols):
    """Merge the LUV and new runs on ``split_geo_id``, keeping capacity and target-year columns."""
    keep = [f'{cap_prefix}netcapacity', f'{cap_prefix}capshare']
    out = None
    for run, df in [('luv', luv_df), ('new', new_df)]:
        nosplit_col = 'geo_id' if 'geo_id' in df.columns else 'nosplit_geo_id'
        cols = ['subreg_id', nosplit_col, 'is_tod', 'name'] + keep + [c for c in year_cols if c in df.columns]
        part = df[cols].rename(columns={'subreg_id': 'split_geo_id', nosplit_col: 'nosplit_geo_id'})
        part = part.rename(columns={c: f'{c}_{run}' for c in part.columns if c != 'split_geo_id'})
        out = part if out is None else out.merge(part, on='split_geo_id', how='outer')
    for col in ['nosplit_geo_id', 'is_tod', 'name']:
        out[col] = out[f'{col}_new'].fillna(out[f'{col}_luv'])
    metadata = {'split_geo_id', 'nosplit_geo_id', 'is_tod', 'name', 'nosplit_geo_id_luv',
                'nosplit_geo_id_new', 'is_tod_luv', 'is_tod_new', 'name_luv', 'name_new'}
    ordered = [c for c in out.columns if c not in metadata]
    return out[['nosplit_geo_id', 'is_tod', 'split_geo_id', 'name'] + ordered].sort_values('split_geo_id', ignore_index=True)


hh_compare = merge_runs(luv_hh, new_hh, 'DU', ['HH2044', 'HH2050'])
emp_compare = merge_runs(luv_emp, new_emp, 'EMP', ['Emp2044', 'Emp2050'])

OUT_FILE = Path('output/hct_controls_luv_vs_new_comparison.xlsx')
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUT_FILE) as writer:
    hh_compare.to_excel(writer, sheet_name='HH', index=False)
    emp_compare.to_excel(writer, sheet_name='Emp', index=False)

print(f'wrote {OUT_FILE.resolve()}')
hh_compare.head()

wrote C:\Users\jkolberg\PythonProjects\PSRC\control_totals\examples\summer_2026\output\luv_vs_new_comparison.xlsx


,nosplit_geo_id,is_tod,split_geo_id,name,DUnetcapacity_luv,DUcapshare_luv,HH2050_luv,DUnetcapacity_new,DUcapshare_new,HH2044_new,HH2050_new
0,1.0,0.0,1,Bellevue,10034,17.1,36370,71912.0,55.2,42039.0,42924.0
1,2.0,0.0,2,Seattle,44026,15.7,104262,149317.0,23.5,145154.0,147514.0
2,3.0,0.0,3,Auburn,16361,64.5,26281,44128.0,64.3,26798.0,27798.0
3,4.0,0.0,4,Bothell,5577,50.5,11550,49106.0,80.1,11613.0,12563.0
4,5.0,0.0,5,Burien,4175,28.2,12704,34634.0,35.4,14082.0,14278.0


In [ ]:
def compare_hct_split(old,new,year):
    # comparison of summer 2026 with unlimited capacity
    old_hh = pd.read_excel(f'output/LUVit_ct_by_tod_generator-2026-{old}_90-90-90.xlsx', sheet_name='HHwork').query(rgid_mask)
    old_emp = pd.read_excel(f'output/LUVit_ct_by_tod_generator-2026-{old}_90-90-90.xlsx', sheet_name='EMPwork').query(rgid_mask)

    new_hh = pd.read_excel(f'output/LUVit_ct_by_tod_generator-2026-{new}_90-90-90.xlsx', sheet_name='HHwork').query(rgid_mask)
    new_emp = pd.read_excel(f'output/LUVit_ct_by_tod_generator-2026-{new}_90-90-90.xlsx', sheet_name='EMPwork').query(rgid_mask)
    hh_comparison = old_hh.merge(new_hh, on=['subreg_id','is_tod','name'], suffixes=(f'_{old}', f'_{new}'))
    hh_comparison['HH_diff'] = hh_comparison[f'HH{year}_{new}'] - hh_comparison[f'HH{year}_{old}']
    emp_comparison = old_emp.merge(new_emp, on=['subreg_id','is_tod','name'], suffixes=(f'_{old}', f'_{new}'))
    emp_comparison['Emp_diff'] = emp_comparison[f'Emp{year}_{new}'] - emp_comparison[f'Emp{year}_{old}']

    with pd.ExcelWriter(f'output/ct_by_hct_comparison_{old}_to_{new}.xlsx') as writer:
        hh_comparison[['subreg_id','is_tod','name',f'HH{year}_{old}',f'HH{year}_{new}','HH_diff']].to_excel(writer,sheet_name='HH', index=False)
        emp_comparison[['subreg_id','is_tod','name',f'Emp{year}_{old}',f'Emp{year}_{new}','Emp_diff']].to_excel(writer,sheet_name='EMP', index=False)

    return hh_comparison, emp_comparison

In [26]:
hh_comparison, emp_comparison = compare_hct_split('10-13', '10-14', 2050)

#### compare to LUV.3 official run

In [15]:
luv_path = "J:/Projects/LandUseVision/LUV.3/Control_totals/SubJuris_ConTots/LUVit_ct_by_tod_generator_90-90-90_2024-10-18_6575_LUVit_Match.xlsx"
update26_path = 'output/LUVit_ct_by_tod_generator-2026-10-28_90-90-90.xlsx'

In [16]:
target_xwalk_luv = (
    pd.read_excel("J:/Projects/LandUseVision/LUV.3/Final_Products/LUV-it Dataset.xlsx", sheet_name="Control + HCT Geographies")
)[['control_hct_id','control_id','target_id','control_hct_name','County']]

In [17]:
target_xwalk26 = pd.read_csv('data/control_target_xwalk.csv')

In [20]:
import numpy as np

OUT_FILE = Path('output/ct_luv_vs_update26_comparison.xlsx')
COUNTY_NAMES = {53033: 'King', 53035: 'Kitsap', 53053: 'Pierce', 53061: 'Snohomish'}
LUV_BASE, UPDATE_BASE = 2020, 2023
COMPARE_YEARS = [2044, 2050]
BASE_YEARS = {'luv': LUV_BASE, 'update26': UPDATE_BASE}
PATHS = {'luv': luv_path, 'update26': update26_path}
MEASURES = {'HH': ('HHwork', 'HH'), 'EMP': ('EMPwork', 'Emp')}  # label: (work sheet, one-column-per-year sheet)
GEO_COLS = ['county', 'control_id', 'target_id', 'control_name']
STATUS = {'both': 'in both', 'left_only': 'dropped in update26', 'right_only': 'added in update26'}

# unchanged work tables
raw_tables = {(run, label): pd.read_excel(PATHS[run], sheet_name=ws)
              for run in PATHS for label, (ws, _) in MEASURES.items()}

# control-level lookup: update26 xwalk preferred, luv xwalk fills controls that no longer exist
_cols = ['control_id', 'target_id', 'control_name', 'county']
_luv = (target_xwalk_luv.rename(columns={'control_hct_name': 'control_name', 'County': 'county'})
        .drop_duplicates('control_id')[_cols])
_upd = target_xwalk26.assign(county=target_xwalk26['county_id'].map(COUNTY_NAMES))[_cols]
control_lookup = (_upd.set_index('control_id').combine_first(_luv.set_index('control_id'))
                  .reset_index().astype({'target_id': 'Int64'}))


def run_data(run, label):
    """subreg_id -> control_id/is_tod ids and the base, 2044 and 2050 values of one run and measure."""
    work, base = raw_tables[(run, label)], BASE_YEARS[run]
    geo_col = 'geo_id' if 'geo_id' in work.columns else 'nosplit_geo_id'
    ids = work[['subreg_id', geo_col, 'is_tod']].rename(columns={geo_col: 'control_id'})
    vals = pd.read_excel(PATHS[run], sheet_name=MEASURES[label][1])
    vals.columns = ['subreg_id'] + [int(c) for c in vals.columns[1:]]
    years = [base, *COMPARE_YEARS]
    vals = vals[['subreg_id', *years]].rename(columns={y: f'{run}_{y}' for y in years})
    return ids, vals


def add_metrics(df):
    """Differences and average annual growth from each run's own base year."""
    for y in COMPARE_YEARS:
        df[f'diff_{y}'] = df[f'update26_{y}'] - df[f'luv_{y}']
        df[f'pct_diff_{y}'] = (df[f'diff_{y}'] / df[f'luv_{y}'].replace(0, np.nan) * 100).round(1)
    return df


def value_cols():
    return [f'{run}_{y}' for run, base in BASE_YEARS.items() for y in [base, *COMPARE_YEARS]]


def order_columns(df, lead):
    cols = [f'luv_{LUV_BASE}', f'update26_{UPDATE_BASE}']
    for y in COMPARE_YEARS:
        cols += [f'luv_{y}', f'update26_{y}', f'diff_{y}', f'pct_diff_{y}']
    return df[lead + cols].reset_index(drop=True)


def per_control(ids, vals):
    return (vals.merge(ids[['subreg_id', 'control_id']], on='subreg_id').drop(columns='subreg_id')
            .groupby('control_id', as_index=False).sum(min_count=1))


def build_comparisons(label):
    luv_ids, luv_vals = run_data('luv', label)
    upd_ids, upd_vals = run_data('update26', label)
    ids = pd.concat([upd_ids, luv_ids]).drop_duplicates('subreg_id')

    sub = luv_vals.merge(upd_vals, on='subreg_id', how='outer', indicator='status')
    sub['status'] = sub['status'].map(STATUS)
    sub = sub.merge(ids, on='subreg_id', how='left').merge(control_lookup, on='control_id', how='left')
    sub = add_metrics(sub).sort_values(['county', 'target_id', 'control_id', 'subreg_id'])
    by_subreg = order_columns(sub, ['subreg_id', *GEO_COLS, 'is_tod', 'status'])

    ctl = per_control(luv_ids, luv_vals).merge(per_control(upd_ids, upd_vals), on='control_id',
                                               how='outer', indicator='status')
    ctl['status'] = ctl['status'].map(STATUS)
    ctl = ctl.merge(control_lookup, on='control_id', how='left')
    ctl_totals = ctl.copy()
    ctl = add_metrics(ctl).sort_values(['county', 'target_id', 'control_id'])
    by_control = order_columns(ctl, [*GEO_COLS, 'status'])

    cty = ctl_totals.groupby('county', as_index=False)[value_cols()].sum(min_count=1)
    reg = pd.DataFrame([{'county': 'Region', **ctl_totals[value_cols()].sum(min_count=1).to_dict()}])
    by_county = order_columns(add_metrics(pd.concat([cty, reg], ignore_index=True)), ['county'])
    return by_subreg, by_control, by_county


def build_geo_changes(comparisons):
    """Added / dropped geographies with the 2050 value from whichever run has the geography."""
    keys = ['level', 'id', *GEO_COLS, 'status']
    by_measure = []
    for label, (by_subreg, by_control, _) in comparisons.items():
        frames = []
        for level, df in [('subreg_id', by_subreg), ('control_id', by_control)]:
            ch = df[df['status'] != 'in both'].copy()
            ch['level'] = level
            ch['id'] = ch[level]
            ch[f'{label}_2050'] = ch['update26_2050'].fillna(ch['luv_2050'])
            frames.append(ch[[*keys, f'{label}_2050']])
        by_measure.append(pd.concat(frames))
    out = by_measure[0]
    for df in by_measure[1:]:
        out = out.merge(df, on=keys, how='outer')
    return out.sort_values(['level', 'status', 'county', 'id'], ascending=[False, True, True, True]).reset_index(drop=True)

def pretty_columns(df):
    """Readable headers for the comparison tabs; source tabs keep their original column names."""
    fixed = {'subreg_id': 'Subreg ID', 'control_id': 'Control ID', 'target_id': 'Target ID',
             'control_name': 'Control Name', 'is_tod': 'Is TOD', 'county': 'County', 'status': 'Status',
             'level': 'Level', 'id': 'ID'}
    prefixes = {'luv_': 'LUV ', 'update26_': 'Update26 ', 'diff_': 'Diff ', 'pct_diff_': '% Diff ',
                'HH_': 'HH ', 'EMP_': 'EMP '}

    def rename(col):
        if col in fixed:
            return fixed[col]
        for prefix, label in prefixes.items():
            if col.startswith(prefix):
                return label + col[len(prefix):]
        return col

    return df.rename(columns=rename)

def format_sheet(ws):
    from openpyxl.styles import Alignment, Font
    from openpyxl.utils import get_column_letter
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(wrap_text=True, vertical='top')
    for i, col in enumerate(ws.iter_cols(min_row=1, max_row=min(ws.max_row, 200)), start=1):
        longest = max(len(str(c.value)) if c.value is not None else 0 for c in col[1:] or col)
        ws.column_dimensions[get_column_letter(i)].width = min(max(longest + 2, 10), 32)
    ws.row_dimensions[1].height = 45


comparisons = {label: build_comparisons(label) for label in MEASURES}
geo_changes = build_geo_changes(comparisons)

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUT_FILE, engine='openpyxl') as writer:
    for (run, label), df in raw_tables.items():
        df.to_excel(writer, sheet_name=f'{run}_{MEASURES[label][0]}', index=False)
    for suffix, i in [('by_subreg', 0), ('by_control', 1), ('by_county', 2)]:
        for label, frames in comparisons.items():
            pretty_columns(frames[i]).to_excel(writer, sheet_name=f'{label}_{suffix}', index=False)
    pretty_columns(geo_changes).to_excel(writer, sheet_name='geo_changes', index=False)
    for ws in writer.book.worksheets:
        format_sheet(ws)

print(f'wrote {OUT_FILE.resolve()}')
geo_changes

wrote C:\Users\jkolberg\PythonProjects\PSRC\control_totals\examples\summer_2026\output\ct_luv_vs_update26_comparison.xlsx


,level,id,county,control_id,target_id,control_name,status,HH_2050,EMP_2050
0,subreg_id,1019,King,19,19,Newcastle,dropped in update26,2447.0,1792.0
1,subreg_id,1024,King,24,24,Renton PAA - West Hill,dropped in update26,1560.0,1080.0
2,subreg_id,1026,King,26,26,Woodinville,dropped in update26,1707.0,8172.0
3,subreg_id,1073,Kitsap,73,73,Poulsbo,dropped in update26,538.0,3619.0
4,subreg_id,1074,Kitsap,74,74,Poulsbo PUTA,dropped in update26,44.0,2.0
5,subreg_id,1085,Pierce,85,85,"Mid-County, Parkland-Spanaway-Midland, South Hill",dropped in update26,29310.0,25777.0
6,subreg_id,175,Snohomish,175,175,Woodway MUGA,dropped in update26,142.0,39.0
7,subreg_id,405,Pierce,405,107,JBLM - UGA,added in update26,3402.0,7729.0
8,control_id,175,Snohomish,175,175,Woodway MUGA,dropped in update26,142.0,39.0
9,control_id,405,Pierce,405,107,JBLM - UGA,added in update26,3402.0,7729.0
